In [ ]:
from llm.google_llm import GoogleLLM
import json

api_key = ""
with open("cred.json", "r") as f:
	cred = json.load(f)
	api_key = cred["google_api_key"]

llm = GoogleLLM(model_id="gemini-2.5-flash", api_key=api_key)

In [ ]:
from prompts import builder
from llm.llm import GenerationResult, GenerationParams

system_instruction, prompt = builder.build_prompt("rough_outline",
    {
		"detective": "an aging former police detective, now an occasional private investigator with sarcastic wit",
		"tropes": "murder mystery, closed room mystery, two points of view",
		"constraints": "no supernatural elements, no random hidden rooms",
		"setting": "2020s, a small rural town, an old mansion on a hill",
		"tone": "tense and suspenseful, but humorous and witty"
	}
)

generation_params = GenerationParams(
    max_tokens=10000,
    temperature=0.9,
    top_p=0.9,
    top_k=20,
)

ro_response = llm.generate_stream(prompt, system_instruction, generation_params)
rough_outline = ro_response.text

## Rough Outline: The Quiet Poison of Blackwood Manor

### 1. Title
**The Quiet Poison of Blackwood Manor**

### 2. Main Characters

*   **Detective/Protagonist:** **Arthur "Art" Finch**
    *   **Type:** Aging (early 60s) former big-city homicide detective, now a semi-retired, occasional private investigator.
    *   **Personality:** Sharp, cynical, and armed with a bone-dry, often self-deprecating wit. Prefers solitude but possesses a deep, quiet sense of justice. He’s seen too much to be shocked, but still has a moral compass. Divorced, lives with a perpetually unimpressed cat.
    *   **Motivation:** Initially reluctant to take the case, drawn in by the challenge of the "impossible" and a lingering desire to prove his worth after a career-ending incident that wasn't his fault.
*   **Second Point-of-View Character:** **Elara Vance**
    *   **Type:** A bright, ambitious young (late 20s) freelance architect specializing in historical preservation.
    *   **Personality:** Observant, 

In [ ]:
from serialization import StoryDirectory
from utils import information_extraction

title = information_extraction.extract_title(rough_outline)
print(title)
story_directory = StoryDirectory.new(title)
story_directory.save_stage(
    stage="rough_outline",
    prompt=prompt,
    response=ro_response,
    system_instruction=system_instruction,
    model=llm.model_id,
    generation_params=generation_params
)

NameError: name 'rough_outline' is not defined

In [2]:
from schemas.rough_outline import RoughOutline
from prompts import builder
from llm.llm import GenerationParams

system_instruction, prompt = builder.build_prompt("rough_outline",
	{
		"detective": "an aging former police detective, now an occasional private investigator with sarcastic wit",
		"tropes": "murder mystery, closed room mystery, two point of view characters",
		"constraints": "no supernatural elements, no random hidden rooms",
		"setting": "2020s, a small rural town, an old mansion on a hill",
		"tone": "tense and suspenseful, but humorous and witty"
	}
)

generation_params = GenerationParams(
    max_tokens=10000,
    temperature=1,
    top_p=0.9,
    top_k=20,
    response_type="application/json",
    response_schema=RoughOutline,
)

ro_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
rough_outline = RoughOutline.model_validate_json(ro_response.text)

In [20]:
from serialization import StoryDirectory

story_directory = StoryDirectory.new(rough_outline.title)
story_directory.save_stage(
	stage=f"rough_outline",
	prompt=prompt,
	response=ro_response,
	system_instruction=system_instruction,
	model=llm.model_id,
	generation_params=generation_params
)

In [ ]:
from schemas.timeline import Timeline

system_instruction, prompt = builder.build_prompt("timeline",
	{
		"rough_outline": rough_outline
	}
)

generation_params = GenerationParams(
	max_tokens=10000,
	temperature=0.7,
	top_p=0.9,
	top_k=20,
	response_type="application/json",
	response_schema=Timeline,
)

tl_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
timeline = Timeline.model_validate_json(tl_response.text)

points=[TimePoint(time='Decades Ago (approx. 1990s)', description="Eleanor Vance's younger sister dies in a tragic incident, which Godfrey Thorne actively covers up, manipulating events to protect his family's reputation. Eleanor, consumed by grief and a thirst for justice, begins her long-term plan for revenge, continuing her service as the Thorne family's housekeeper, meticulously observing and planning."), TimePoint(time='Several Months Ago', description="Marcus Thorne's gambling debts escalate significantly, making him increasingly desperate for funds and a greater share of the family inheritance."), TimePoint(time='Three Weeks Ago', description='Evelyn Thorne has a very public, heated altercation with her father, Godfrey, in Oakhaven town square, concerning his control over her finances and personal life. This incident makes her resentment widely known.'), TimePoint(time='Two Weeks Ago', description='Clara Vance, an ambitious young journalist, arrives in Oakhaven, eager to make a 

In [ ]:
import json

print(json.dumps(timeline.model_dump(), indent=4))

{
    "points": [
        {
            "time": "Decades Ago (approx. 1990s)",
            "description": "Eleanor Vance's younger sister dies in a tragic incident, which Godfrey Thorne actively covers up, manipulating events to protect his family's reputation. Eleanor, consumed by grief and a thirst for justice, begins her long-term plan for revenge, continuing her service as the Thorne family's housekeeper, meticulously observing and planning."
        },
        {
            "time": "Several Months Ago",
            "description": "Marcus Thorne's gambling debts escalate significantly, making him increasingly desperate for funds and a greater share of the family inheritance."
        },
        {
            "time": "Three Weeks Ago",
            "description": "Evelyn Thorne has a very public, heated altercation with her father, Godfrey, in Oakhaven town square, concerning his control over her finances and personal life. This incident makes her resentment widely known."
        },

In [ ]:
system_instruction, prompt = builder.build_prompt("detailed_outline", 
	{
		"rough_outline": rough_outline,
	}
)

generation_params = GenerationParams(
	max_tokens=15000,
	temperature=1,
	top_p=0.9,
	top_k=20,
)

do_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
detailed_outline = do_response.text

Here is a detailed, chapter-by-chapter outline for "The Architect of Silence":

1.  **Chapter 1: The Weight of Silence**
    Silas Thorne, a private investigator whose office overlooks the perpetually damp streets of Veridian City, sits amidst cold case files, haunted by the ghost of his murdered wife, Lena. His solitude is broken by a call from Jonathan Vance, Eleanor's estranged father, whose clipped tones betray a desperate man. Vance recounts the police report of his daughter’s murder: a seemingly professional hit in her high-security apartment, but with a baffling detail – a window latch secured with a sailor’s knot, intricate and anachronistic, and Eleanor’s body positioned with almost ceremonial precision. This specific detail triggers a painful, visceral memory for Silas, an echo of a similar, inexplicable anomaly at Lena’s own crime scene, compelling him to take the case despite his initial reluctance.

2.  **Chapter 2: An Architect's Blueprint**
    Silas uses his old, clande

In [ ]:
story_directory.save_stage(
	stage="detailed_outline",
	prompt=prompt,
	response=do_response,
	system_instruction=system_instruction,
	model=llm.model_id,
	generation_params=generation_params
)

In [14]:
from schemas.detailed_outline import DetailedOutline

system_instruction, prompt = builder.build_prompt("detailed_outline", 
	{
		"rough_outline": rough_outline,
	}
)

generation_params = GenerationParams(
    max_tokens=15000,
	temperature=1,
	top_p=0.9,
	top_k=20,
	response_type="application/json",
	response_schema=DetailedOutline,
)

do_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
detailed_outline = DetailedOutline.model_validate_json(do_response.text)

In [21]:

print(story_directory.path)
story_directory.save_stage(
	stage=f"detailed_outline",
	prompt=prompt,
	response=do_response,
	system_instruction=system_instruction,
	model=llm.model_id,
	generation_params=generation_params
)

Stories\the_raven's_peak_inheritance_2025-12-08_125252


In [ ]:
import time

chapter_outlines = [ch for ch in (c.strip() for c in detailed_outline.split("\n\n")) if ch and len(ch) > 100]

chapters = []
for i, chapter_outline in enumerate(chapter_outlines):
    system_instruction, prompt = builder.build_prompt("chapter",
		{
			"rough_outline": rough_outline,
			"previous_chapter": chapter_outlines[i-1] if i > 0 else "N/A",
			"next_chapter": chapter_outlines[i+1] if i < len(chapter_outlines) - 1 else "N/A",
			"current_chapter": chapter_outline,
			"index": i + 1
		}
	)
    generation_params = GenerationParams(
		temperature=1,
		top_p=0.9,
		top_k=20,
	)
    print(f"\n\n--- Generating Chapter {i+1} ---\n\n")
    ch_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
    chapter = ch_response.text
    chapters.append(chapter)
    story_directory.save_stage(
		stage=f"chapter_{i+1:02d}",
		prompt=prompt,
		response=ch_response,
		system_instruction=system_instruction,
		model=llm.model_id,
		generation_params=generation_params
	)
    if (i + 1) % 2 == 0:
        print("\n\n--- Pausing for 30 seconds to avoid rate limits ---\n\n")
        time.sleep(30)  # Pause to avoid rate limits



--- Generating Chapter 1 ---


The perpetually damp air of Veridian City clung to everything, seeping into the old brickwork, blurring the edges of the gleaming skyscrapers, and settling like a shroud over Silas Thorne’s third-story office. Outside his window, the late autumn drizzle was a ceaseless whisper against the glass, a melancholic soundtrack to a life lived in shades of grey. Inside, the silence was heavier, broken only by the rhythmic hum of an ancient radiator and the turning of pages in cold case files. Silas sat hunched over his desk, the lamplight casting long shadows that distorted the familiar stacks of folders into monstrous shapes. Each file was a tombstone, each unsolved mystery a shard of glass in his own shattered past. He moved through his days like a ghost haunting his own life, a life irrevocably cleaved into 'before' and 'after' by the brutal, inexplicable murder of his wife, Lena. Her memory was a constant, searing ache, a quiet sentinel in the desolate land

TypeError: can only concatenate str (not "NoneType") to str

In [22]:
import time

chapters = []
for i, chapter_outline in enumerate(detailed_outline.chapters):
    system_instruction, prompt = builder.build_prompt("chapter",
		{
			"rough_outline": rough_outline,
			"previous_chapter": detailed_outline.chapters[i-1] if i > 0 else "N/A",
			"next_chapter": detailed_outline.chapters[i+1] if i < len(detailed_outline.chapters) - 1 else "N/A",
			"current_chapter": chapter_outline,
			"index": i + 1
		}
	)
    generation_params = GenerationParams(
        max_tokens=15000,
		temperature=1,
		top_p=0.9,
		top_k=20,
	)
    print(f"\n\n--- Generating Chapter {i+1} ---\n\n")
    ch_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
    chapter = ch_response.text
    chapters.append(chapter)
    story_directory.save_stage(
		stage=f"chapter_{i+1:02d}",
		prompt=prompt,
		response=ch_response,
		system_instruction=system_instruction,
		model=llm.model_id,
		generation_params=generation_params
	)
    if (i + 1) % 2 == 0:
        print("\n\n--- Pausing for 30 seconds to avoid rate limits ---\n\n")
        time.sleep(30)  # Pause to avoid rate limits



--- Generating Chapter 1 ---




--- Generating Chapter 2 ---




--- Pausing for 30 seconds to avoid rate limits ---




--- Generating Chapter 3 ---




--- Generating Chapter 4 ---




--- Pausing for 30 seconds to avoid rate limits ---




--- Generating Chapter 5 ---




--- Generating Chapter 6 ---




--- Pausing for 30 seconds to avoid rate limits ---




--- Generating Chapter 7 ---




--- Generating Chapter 8 ---




--- Pausing for 30 seconds to avoid rate limits ---




--- Generating Chapter 9 ---




--- Generating Chapter 10 ---




--- Pausing for 30 seconds to avoid rate limits ---




--- Generating Chapter 11 ---




--- Generating Chapter 12 ---




--- Pausing for 30 seconds to avoid rate limits ---




In [23]:
story_directory.save_plain_text(
	stage="full_story",
	text="\n\n".join(f"{index}: {name}\n{chapter}" for index, name, chapter in zip([f"Chapter {i+1}" for i in range(len(chapters))], [chapter_summary.title for chapter_summary in detailed_outline.chapters], chapters))
)

In [35]:
from schemas.chunk_evaluation import  ChapterChunkEvaluation
from serialization import StoryDirectory

story_directory = StoryDirectory.open("the_raven's_peak_inheritance_2025-12-08_125252")

def evaluate_chapter_pair(id1 : int, id2 : int) -> ChapterChunkEvaluation:
	chapter1 = story_directory.load_stage(f"chapter_{id1:02d}")["response"]["text"]
	chapter2 = story_directory.load_stage(f"chapter_{id2:02d}")["response"]["text"]

	system_instruction, prompt = builder.build_prompt("chunk_evaluation",
		{
			"chapter1": f"Chapter {id1}:\n{chapter1}",
			"chapter2": f"Chapter {id2}:\n{chapter2}"
		}
	)
	generation_params = GenerationParams(
		max_tokens=15000,
		temperature=1,
		top_p=0.9,
		top_k=20,
		response_type="application/json",
		response_schema=ChapterChunkEvaluation,
	)

	ce_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params, print_output=True)
	evaluation = ChapterChunkEvaluation.model_validate_json(ce_response.text)
 
	story_directory.save_stage(
		stage=f"chunk_evaluation_{id1:02d}_{id2:02d}",
		prompt=prompt,
		response=ce_response,
		system_instruction=system_instruction,
		model=llm.model_id,
		generation_params=generation_params
	)
 
	return evaluation

In [36]:
evaluate_chapter_pair(1, 2)
evaluate_chapter_pair(3, 4)
evaluate_chapter_pair(5, 6)
evaluate_chapter_pair(7, 8)
evaluate_chapter_pair(9, 10)
evaluate_chapter_pair(11, 12)
evaluate_chapter_pair(13, 14)

{
  "summary": {
    "chapter_numbers": [1, 2],
    "short_summary": "Art Blackwood, a cynical retired detective, is called by Sheriff Brody to investigate the locked-room murder of the universally disliked Godfrey Thorne at Raven's Peak Manor. En route, Art reluctantly picks up tenacious journalist Clara Vance, and both become trapped at the isolated manor by a sudden blizzard. Inside the meticulously preserved study, Godfrey is found stabbed with a letter opener, with no apparent entry or exit points. Art begins interviewing the immediate suspects: Godfrey's resentful daughter Evelyn, ambitious son Marcus, and stoic housekeeper Eleanor, all of whom offer vague alibis.",
    "key_events": [
      "Sheriff Brody calls Art Blackwood to investigate Godfrey Thorne's death.",
      "Art learns Godfrey was found dead in a locked room with no forced entry.",
      "Art encounters and gives a ride to journalist Clara Vance, who is also heading to the manor.",
      "A severe blizzard cuts off

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 41.253243535s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.

In [ ]:
from schemas.global_evaluation import GlobalStoryEvaluation
from serialization import StoryDirectory

story_directory = StoryDirectory.open("The_Architect_of_Silence_2025-11-27_174658")

def evaluate_global(chapters_count : int) -> GlobalStoryEvaluation:
	summaries = []
	for i in range(1, chapters_count, 2):
		chunk_summary = story_directory.load_stage(f"chapter_evaluation_{i:02d}_{i+1:02d}")["response"].text
		summaries.append(chunk_summary)
    
	system_instruction, prompt = builder.build_prompt("global_evaluation",
        {
			"summaries": summaries
		}
	)
 
	generation_params = GenerationParams(
		temperature=1,
		top_p=0.9,
		top_k=20,
		response_type="application/json",
		response_schema=GlobalStoryEvaluation,
	)

	ge_response = llm.generate_stream(prompt, system_instruction=system_instruction, generation_params=generation_params)
	evaluation = GlobalStoryEvaluation.model_validate_json(ge_response.text)
 
	story_directory.save_stage(
		stage=f"global_story_evaluation",
		prompt=prompt,
		response=ge_response,
		system_instruction=system_instruction,
		model=llm.model_id,
		generation_params=generation_params
	)
	return evaluation

In [28]:
evaluate_global(14)

{
  "overall_coherence_score": 8,
  "detective_logic_strength_score": 9,
  "twist_originality_score": 7,
  "clue_payoff_quality_score": 9,
  "suspect_motivation_strength_score": 9,
  "fair_play_rule_respect_score": 8,
  "final_resolution_strength_score": 9,
  "major_plot_holes": [
    "The ease with which Silas bypasses 'impenetrable' high-tech security in Eleanor's apartment and later an antique lock at Albright's sprawling estate feels too convenient, undermining the described security levels.",
    "The rapid and convenient pinpointing of a specific, obscure book within a vast, rarely visited archive by Anya, based on general historical society information, strains credulity.",
    "Dr. Albright's immediate and detailed confession of major crimes, including murder, to Silas before any undeniable, legally admissible evidence is presented (beyond Silas's archival discovery) is highly improbable for a seasoned criminal mastermind.",
    "The lone officer on watch at a high-profile murd